# `SmoothedSurface`

`SmoothedSurface` removes short-wavelength geometric roughness from a discrete surface using Taubin smoothing. It is particularly useful for closed surfaces, where ordinary Laplacian smoothing tends to produce systematic shrinkage.

For normal use, the main parameter is simply `cutoff_wavelength`. The iteration count and Taubin coefficients are determined automatically.


## Setup


In [ ]:
import numpy as np
import pyvista as pv
import nematics3d as n3d


## Example: a noisy closed surface

We first construct a triangulated sphere and add short-wavelength radial roughness. In real use, this would normally be replaced by your reconstructed or measured surface mesh.


In [ ]:
surface = pv.Sphere(
    radius=10.0,
    theta_resolution=80,
    phi_resolution=80,
).triangulate()

points = np.asarray(surface.points, dtype=float)
radius = np.linalg.norm(points, axis=1)
direction = points / radius[:, None]
x, y, z = direction.T

roughness = 0.25 * np.sin(18.0 * np.arctan2(y, x)) * (1.0 - z**2)

surface_noisy = surface.copy()
surface_noisy.points = points + roughness[:, None] * direction


## Basic use

The shortest useful call is:

```python
smoothed = n3d.SmoothedSurface(
    surface_noisy,
    cutoff_wavelength=2.0,
)
```

`cutoff_wavelength` is a **physical wavelength in the same length unit as the surface coordinates**. It is not a mesh spacing, a displacement distance, or an iteration count.


In [ ]:
smoothed = n3d.SmoothedSurface(
    surface_noisy,
    cutoff_wavelength=2.0,
)


## Getting the result

The main output is `result`, a smoothed `pyvista.PolyData` surface:


In [ ]:
surface_smooth = smoothed.result
surface_smooth


If only the final vertex coordinates are needed, use `vertices`:


In [ ]:
vertices_smooth = smoothed.vertices
print(vertices_smooth.shape)
print(vertices_smooth.flags.writeable)


`vertices` has shape `(n_points, 3)` and is exposed as a read-only NumPy array. `result` should normally be preferred when the surface topology is also needed.


## Visual comparison


In [ ]:
plotter = pv.Plotter(shape=(1, 2))

plotter.subplot(0, 0)
plotter.add_text("Input")
plotter.add_mesh(surface_noisy, smooth_shading=True)

plotter.subplot(0, 1)
plotter.add_text("Smoothed")
plotter.add_mesh(surface_smooth, smooth_shading=True)

plotter.link_views()
plotter.show()


## Meaning of `cutoff_wavelength`

The cutoff is defined from the complete smoothing pass, not from one Taubin iteration. Let

$$
\kappa_c = \left(\frac{2\pi}{\ell_c}\right)^2,
$$

where $\ell_c$ is `cutoff_wavelength`. Nematics3D chooses the filter so that

$$
G_N(\kappa_c)=\frac{1}{\sqrt{2}}.
$$

Therefore a mode whose wavelength is exactly `cutoff_wavelength` retains $1/\sqrt{2}$ of its original amplitude after the complete smoothing operation. Roughly speaking:

- wavelengths much longer than the cutoff are preserved more strongly;
- the cutoff wavelength is reduced to $1/\sqrt{2}$ amplitude;
- wavelengths shorter than the cutoff are suppressed more strongly.

The cutoff should therefore be chosen from the physical scale of the roughness you want to remove. There is intentionally no geometry-independent default.


## Options

`SmoothedSurface` currently exposes only two smoothing options:

- `cutoff_wavelength`: required physical cutoff wavelength;
- `taubin_ratio`: the dimensionless ratio $r=-\mu/\lambda$.

Normally only `cutoff_wavelength` should be changed. `taubin_ratio` defaults to `1.0674`. The raw coefficients `lambda`, `mu`, and the number of Taubin pairs are deliberately not user options because they depend on both the requested physical cutoff and the discrete spectrum of the mesh.


In [ ]:
smoothed.opts


Options can also be created explicitly:


In [ ]:
opts = n3d.OptsSmoothedSurface(
    cutoff_wavelength=2.0,
)

smoothed_from_opts = n3d.SmoothedSurface(
    surface_noisy,
    opts=opts,
)


## Changing the smoothing scale

The smoothing scale can be changed after construction using the normal commit interface:


In [ ]:
smoothed.act_commit(cutoff_wavelength=2.5)


When only smoothing options change, `SmoothedSurface` reuses the fixed Laplace--Beltrami operator constructed from the current raw surface. It recomputes the filter parameters and starts the new smoothing pass again from the original triangulated geometry rather than smoothing the previous result repeatedly.

This avoids cumulative smoothing and also preserves the spectral interpretation of the filter.


## Public diagnostic attributes

Most applications only need `result`, `vertices`, and the values in `opts`. `SmoothedSurface` nevertheless exposes a small set of `calc_` attributes so that the automatically constructed filter can be inspected and reproduced.

The public diagnostics are:

| Attribute | Meaning |
| --- | --- |
| `calc_kappa_cutoff` | Laplace--Beltrami eigenvalue corresponding to `cutoff_wavelength` |
| `calc_kappa_max` | largest resolved eigenvalue of the discrete mesh operator |
| `calc_iterations` | automatically selected number of Taubin pairs |
| `calc_lambda` | resolved positive Taubin coefficient |
| `calc_mu` | resolved negative Taubin coefficient |
| `calc_is_smoothed` | whether the current smoothing pass completed successfully |
| `calc_status` | human-readable status of the current smoothing result |

Internal mesh connectivity, mass matrices, stiffness matrices, and temporary geometry are implementation details and are intentionally not part of the public `calc_` interface.


In [ ]:
print("kappa cutoff :", smoothed.calc_kappa_cutoff)
print("kappa max    :", smoothed.calc_kappa_max)
print("Taubin pairs :", smoothed.calc_iterations)
print("lambda       :", smoothed.calc_lambda)
print("mu           :", smoothed.calc_mu)
print("is smoothed  :", smoothed.calc_is_smoothed)
print("status       :", smoothed.calc_status)


### `calc_kappa_cutoff`

This is simply the requested physical wavelength expressed in Laplace--Beltrami spectral coordinates:

$$
\kappa_c = \left(\frac{2\pi}{\texttt{cutoff_wavelength}}\right)^2.
$$

It has units of inverse length squared.


### `calc_kappa_max`

The surface discretization defines a generalized eigenvalue problem

$$
K\phi = \kappa M\phi,
$$

where $K$ is the cotangent stiffness matrix and $M$ is the lumped mass matrix. `calc_kappa_max` is the largest resolved $\kappa$ of this discrete operator.

It characterizes the highest spatial frequency represented by the current mesh and is used to make sure the automatically constructed Taubin filter is stable at the high-frequency end of the spectrum. It is a property of the mesh discretization, not an additional smoothing option.


### `calc_iterations`

One Taubin pair consists of two updates,

$$
V \leftarrow V + \lambda L V,
$$

followed by

$$
V \leftarrow V + \mu L V.
$$

`calc_iterations` is the number $N$ of these **pairs**, not the total number of individual Laplacian updates. For example, `calc_iterations == 5` means five positive-$\lambda$ steps and five negative-$\mu$ steps.

Nematics3D chooses the smallest positive $N$ satisfying the high-frequency stability requirement for the requested cutoff and current mesh. This is why iteration count is a diagnostic rather than a user tuning parameter.


### `calc_lambda` and `calc_mu`

These are the two coefficients of the resolved Taubin pair. The convention is

$$
\lambda>0, \qquad \mu<0,
$$

with

$$
-\frac{\mu}{\lambda}=\texttt{taubin_ratio}.
$$

Because the Laplace--Beltrami eigenvalue has units of inverse length squared, both coefficients have units of length squared. They should normally be regarded as derived quantities rather than values to tune directly.


### `calc_is_smoothed` and `calc_status`

These describe the state of the current result. After a successful smoothing pass, `calc_is_smoothed` is `True` and `calc_status` is `"Success"`. They are mainly useful for inspection, logging, and higher-level workflows.


## The filter represented by the diagnostics

The discrete Laplace--Beltrami operator uses the sign convention

$$
L\phi=-\kappa\phi, \qquad \kappa\ge 0.
$$

For one Taubin pair, a mode with eigenvalue $\kappa$ is multiplied by

$$
g(\kappa)=(1-\lambda\kappa)(1-\mu\kappa).
$$

After $N$ pairs, the complete transfer function is

$$
G_N(\kappa)=g(\kappa)^N.
$$

The public diagnostics therefore contain enough information to reconstruct the complete spectral filter used for the current result.


### Verify the cutoff

For example, the following should evaluate to $1/\sqrt{2}$ up to floating-point error:


In [ ]:
kappa_c = smoothed.calc_kappa_cutoff
lambda_ = smoothed.calc_lambda
mu = smoothed.calc_mu
N = smoothed.calc_iterations

gain_at_cutoff = (
    (1.0 - lambda_ * kappa_c)
    * (1.0 - mu * kappa_c)
) ** N

print(gain_at_cutoff)
print(1.0 / np.sqrt(2.0))


## Choosing a useful cutoff

Choose `cutoff_wavelength` from the geometric scale that separates unwanted roughness from meaningful surface structure.

For example, suppose the meaningful shape varies over scales around `10` length units while reconstruction noise produces corrugations around `1`--`2` units. A cutoff of order `2` is then a natural value to test.

The parameter is deliberately physical rather than iteration-based: changing mesh resolution while representing the same physical surface does not change what a wavelength of `2` means, although the discrete spectrum and automatically selected iteration count may change.


## Shrinkage and volume preservation

Ordinary positive Laplacian smoothing behaves similarly to mean-curvature flow and tends to move a curved closed surface inward. The positive/negative Taubin pair strongly suppresses this systematic low-frequency shrinkage.

This is **not** an exact volume constraint. `SmoothedSurface` does not enforce constant enclosed volume or constant area. If exact conservation is required, it should be imposed separately.


## Mesh requirements and errors

The input must be convertible to a non-empty triangle surface. In particular, vertices must be finite, triangles must be non-degenerate, and every retained vertex must participate in valid surface geometry.

The requested cutoff and `taubin_ratio` must also admit a finite stable Taubin filter for the mesh spectrum. If they do not, `SmoothedSurface` raises `SurfaceSmoothingConfigError` rather than silently applying an unstable filter.


## Summary

For ordinary use:

```python
smoothed = n3d.SmoothedSurface(
    surface,
    cutoff_wavelength=desired_physical_scale,
)

surface_smooth = smoothed.result
```

The intended interface is deliberately small:

- use `cutoff_wavelength` to specify the physical smoothing scale;
- use `result` for the final surface and `vertices` for its coordinates;
- leave `taubin_ratio` at its default unless there is a specific reason to change the filter shape;
- inspect the public `calc_` attributes only when filter diagnostics or reproducibility information is needed.

Internal discretization caches are not part of the user-facing API. Surface-function interpolation and sampling are also outside the scope of `SmoothedSurface` itself.
